# Time Series Analysis

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import wfdb
from scipy.signal import butter, filtfilt, find_peaks
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.tools.sm_exceptions import ConvergenceWarning
from timeit import default_timer as timer
from sklearn.preprocessing import StandardScaler

In [ ]:
FOLDER_PATH = Path("../../data/time-series")  

subdirs = sorted([d for d in FOLDER_PATH.iterdir() if d.is_dir()])

print(f"Total directories found: {len(subdirs)}\n")

subject_infos = []

# Extract subject_id and hadm_id (record) from directory names
for subdir in subdirs:
    # Split directory name: format is "subjectid_hadmid"
    parts = subdir.name.split('_')
    if len(parts) >= 2:
        subject_id = parts[0]
        hadm_id = parts[1]  # This is the record/admission ID
        subject_infos.append({
            'subject_id': subject_id,
            'record_name': hadm_id,
            'folder_name': subdir.name
        })

# Convert to DataFrame for easier manipulation
subject_infos_df = pd.DataFrame(subject_infos)

subject_infos_df.shape

In [ ]:
timeseries_dict = {}

for record in subject_infos_df['record_name'].unique():
    record_path = FOLDER_PATH / f"{subject_infos_df[subject_infos_df['record_name'] == record]['folder_name'].values[0]}" / record
    try:
        # Read the entire signal
        signal, fields = wfdb.rdsamp(str(record_path))
        fs = fields['fs']  # Sampling frequency
        
        # Use the actual length of the signal
        num_samples = signal.shape[0]
        num_channels = signal.shape[1]
        duration_seconds = num_samples / fs
        
        # Create time axis for the entire signal
        time_axis = np.arange(num_samples) / fs
        
        # Create a unique time series concatenating all channels sequentially
        # Flatten column-wise: Ch1 samples, then Ch2 samples, then Ch3 samples, etc.
        concatenated_signal = signal.flatten(order='F')  # 'F' for Fortran-style (column-major)
        
        total_length = concatenated_signal.shape[0]
        
        # Calculate start time (in seconds) for each channel in the concatenated signal
        signal_start_times = []
        for i in range(10):
            start_sample = i * num_samples
            start_time = start_sample / fs
            signal_start_times.append({
                'channel_index': i,
                'channel_name': fields['sig_name'][i],
                'start_sample': start_sample,
                'start_time_seconds': start_time,
                'end_sample': start_sample + num_samples - 1,
                'end_time_seconds': (start_sample + num_samples - 1) / fs
            })
        
        # Store in dictionary
        timeseries_dict[record] = {
            'signal': concatenated_signal,
            'fs': fs,
            'num_channels': num_channels,
            'samples_per_channel': num_samples,
            'total_length': total_length,
            'duration_seconds': duration_seconds,
            'signal_names': fields['sig_name'],
            'signal_start_times': signal_start_times
        }
        
        print(f"Record: {record}")
        print(f"  Channels: {num_channels}")
        print(f"  Samples per channel: {num_samples}")
        print(f"  Total concatenated length: {total_length} (= {num_channels} × {num_samples})")
        print(f"  Duration per channel: {duration_seconds:.2f} seconds")
        print(f"  Signal start times:")
        for st in signal_start_times[:3]:  # Show first 3 channels
            print(f"    {st['channel_name']}: starts at sample {st['start_sample']} (time: {st['start_time_seconds']:.2f}s)")
        if num_channels > 3:
            print(f"    ... and {num_channels - 3} more channels")
        print()

    except Exception as e:
        print(f"Could not read record {record}: {e}")

print(f"\nTotal records processed: {len(timeseries_dict)}")

In [ ]:
# Print the frequency of sampling for the first record
first_record = list(timeseries_dict.keys())[0]
fs_first_record = timeseries_dict[first_record]['fs']
print(f"Sampling frequency for record {first_record}: {fs_first_record} Hz")

## Example of ECG Signal

In [ ]:
# For the first subject, plot all the ECG signals in two columns of six plots
first_subject = subject_infos_df.iloc[0]
record_path = FOLDER_PATH / first_subject['folder_name'] / first_subject['record_name']
record = wfdb.rdrecord(str(record_path))
num_signals = record.p_signal.shape[1]
fig, axs = plt.subplots(6, 2, figsize=(15, 20))
for i in range(num_signals):
    ax = axs[i // 2, i % 2]
    ax.plot(record.p_signal[:, i])
    ax.set_title(f'Signal {i+1}')
plt.tight_layout()
plt.show()

# Time Series Understanding

Given a time series, it is interesting to understand the behaviour of the it; to do so we can check for some particular informations:
- **Trends**
- **Seasonality**
- **Cyclicity**

## Noise removal

### Rolling window approach

The first idea to remove noise in the time series is to use a rolling window (with a defeault value of 5), and smooth the curve. 

In [ ]:
# Remove noise in the time series by applying a rolling window average filter
def moving_average(signal, window_size=5):
    """
    Apply moving average filter to a numpy array signal.
    
    Parameters:
    -----------
    signal : numpy.ndarray
        Input signal
    window_size : int
        Size of the moving average window
        
    Returns:
    --------
    pandas.Series
        Filtered signal with NaN at edges
    """
    # Convert to pandas Series for rolling window operations
    signal_series = pd.Series(signal)
    
    # Standardize: (x - mean) / std
    standardized = (signal_series - signal_series.mean()) / signal_series.std()
    
    # Apply rolling window average
    filtered = standardized.rolling(window=window_size, center=True).mean()
    
    return filtered

def moving_std(signal, window_size=5):
    """
    Apply moving standard deviation filter to a numpy array signal.
    
    Parameters:
    -----------
    signal : numpy.ndarray
        Input signal
    window_size : int
        Size of the moving std window
        
    Returns:
    --------
    pandas.Series
        Filtered signal with NaN at edges
    """
    # Convert to pandas Series for rolling window operations
    signal_series = pd.Series(signal)
    
    # Standardize: (x - mean) / std
    standardized = (signal_series - signal_series.mean()) / signal_series.std()
    
    # Apply rolling window standard deviation
    filtered = standardized.rolling(window=window_size, center=True).std()
    
    return filtered


### Low/High-Pass and Band-Pass filters

Another option is to use *Low-Pass*, *High-Pass* and *Band-Pass* filters to remove the noise on ECG signals.
- **Low-Pass filter**: A Low-Pass filter (LPF) is a filter that passes signals with a frequency lower than a selected cutoff frequency and attenuates signals with frequencies higher than the cutoff frequency.
- **High-Pass filter**: A High-Pass filter (HPF) is an electronic filter that passes signals with a frequency higher than a certain cutoff frequency and attenuates signals with frequencies lower than the cutoff frequency.
- **Band-Pass filter**: A Band-Pass filter allows through components in a specified band of frequencies, called its passband but blocks components with frequencies above or below this band. This contrasts with a high-pass filter, which allows through components with frequencies above a specific frequency, and a low-pass filter, which allows through components with frequencies below a specific frequency.

In [ ]:
def low_pass_filter(signal, fs, cutoff=30, order=5):
    """
    Apply a low-pass Butterworth filter to the signal.

    Parameters:
    signal : numpy.ndarray
        Input signal
    fs : float
        Sampling frequency
    cutoff : float
        Cutoff frequency of the filter
    order : int
        Order of the filter
    Returns:
    numpy.ndarray
        Filtered signal
    """
    # Calculate the Nyquist frequency
    nyquist = 0.5 * fs

    # Normalized cutoff frequency
    normal_cutoff = cutoff / nyquist

    # Get the filter coefficients
    b, a = butter(order, normal_cutoff, btype='low', analog=False)

    # Apply the filter
    filtered_signal = filtfilt(b, a, signal)
    return filtered_signal

def high_pass_filter(signal, fs, cutoff=0.5, order=5):
    """
    Apply a high-pass Butterworth filter to the signal.

    Parameters:
    signal : numpy.ndarray
        Input signal
    fs : float
        Sampling frequency
    cutoff : float
        Cutoff frequency of the filter
    order : int
        Order of the filter
    Returns:
    numpy.ndarray
        Filtered signal
    """
    # Calculate the Nyquist frequency
    nyquist = 0.5 * fs

    # Normalized cutoff frequency
    normal_cutoff = cutoff / nyquist

    # Get the filter coefficients
    b, a = butter(order, normal_cutoff, btype='high', analog=False)

    # Apply the filter
    filtered_signal = filtfilt(b, a, signal)
    return filtered_signal

def band_pass_filter(signal, fs, lowcut=0.5, highcut=30, order=5):
    """
    Apply a band-pass Butterworth filter to the signal.

    Parameters:
    signal : numpy.ndarray
        Input signal
    fs : float
        Sampling frequency
    lowcut : float
        Low cutoff frequency of the filter
    highcut : float
        High cutoff frequency of the filter
    order : int
        Order of the filter
    Returns:
    numpy.ndarray
        Filtered signal
    """
    # Calculate the Nyquist frequency
    nyquist = 0.5 * fs

    # Normalized cutoff frequencies
    low = lowcut / nyquist
    high = highcut / nyquist

    # Get the filter coefficients
    b, a = butter(order, [low, high], btype='band')

    # Apply the filter
    filtered_signal = filtfilt(b, a, signal)
    return filtered_signal


### Pan-Tompkins algorithm

In [ ]:
def pt_algorithm(signal, fs):
    """
    Apply the Pan-Tompkins algorithm to detect R-peaks in ECG signals.
    
    Parameters:
    -----------
    signal : numpy.ndarray
        Input ECG signal
    fs : float
        Sampling frequency
        
    Returns:
    --------
    r_peaks : numpy.ndarray
        Indices of detected R-peaks in the signal
    """
    # Step 1: Low-pass filter (0.5-30 Hz)
    lp_filtered = low_pass_filter(signal, fs, cutoff=30, order=5)

    hp_filtered = high_pass_filter(lp_filtered, fs, cutoff=0.5, order=5)
    
    # Step 2: Derivative filter
    derivative_signal = np.diff(hp_filtered)
    
    # Step 3: Squaring
    squared_signal = np.square(derivative_signal)
    
    # Step 4: Moving window integration
    window_size = int(0.150 * fs)  # 150 ms window
    integrated_signal = moving_average(squared_signal, window_size=window_size).fillna(0).values
    
    return integrated_signal

In [ ]:
def QRSpeaks(signal, fs):
    """
    Detect QRS peaks (R-peaks) in Pan-Tompkins processed signal.
    
    Parameters:
    -----------
    signal : numpy.ndarray
        Integrated signal from Pan-Tompkins algorithm
    fs : float
        Sampling frequency
        
    Returns:
    --------
    peaks : numpy.ndarray
        Indices of detected R-peaks
    """
    # Use find_peaks from scipy.signal (already imported)
    peaks, _ = find_peaks(signal, height=np.mean(signal), distance=round(fs*0.200))
    return peaks

## Time series data quality

In [ ]:
# For each subject, take lead II and V5 joined together in a single signal
subj_dict = {}
for subject_id, row in subject_infos_df.iterrows():
    record_name = row['record_name']
    data = timeseries_dict[record_name]
    signal = data['signal']
    fs = data['fs']
    num_samples = data['samples_per_channel']
    
    # Extract lead II (index 1) and V5 (index 4)
    lead_II = signal[1 * num_samples : 2 * num_samples]
    lead_V5 = signal[4 * num_samples : 5 * num_samples]
    
    subject_id = subject_infos_df.iloc[subject_id]['subject_id']
    subj_dict[subject_id] = {
        'II': lead_II,
        'V5': lead_V5,
        'fs': fs
    }

In [ ]:
scaler = StandardScaler()
subject_id = list(subj_dict.keys())[0]
original_II_signal = subj_dict[subject_id]['II']
original_V5_signal = subj_dict[subject_id]['V5']
fs = subj_dict[subject_id]['fs']

scaled_II_signal = scaler.fit_transform(original_II_signal.reshape(-1, 1)).flatten()
scaled_V5_signal = scaler.fit_transform(original_V5_signal.reshape(-1, 1)).flatten()

### Noise cancellation approach 1: Using the rolling window to clean the time series

In [ ]:
mavg_filtered_II = moving_average(scaled_II_signal, window_size=5).fillna(0).values
mavg_filtered_V5 = moving_average(scaled_V5_signal, window_size=5).fillna(0).values

### Noise cancellation approach 2: Using the Band-Pass filter to cancel the noise in the time series

In [ ]:
bp_filtered_II = band_pass_filter(scaled_II_signal, fs, lowcut=0.5, highcut=30, order=5)
bp_filtered_V5 = band_pass_filter(scaled_V5_signal, fs, lowcut=0.5, highcut=30, order=5)

#### Comparison of the two methods

In [ ]:
# Calculate the differences between the filtered and the original (for each lead separately)
diff_ma_II = mavg_filtered_II - scaled_II_signal
diff_ma_V5 = mavg_filtered_V5 - scaled_V5_signal
diff_bp_II = bp_filtered_II - scaled_II_signal
diff_bp_V5 = bp_filtered_V5 - scaled_V5_signal

print(f"Lead II - Difference MA Filter - Mean: {np.mean(diff_ma_II):.4f}, Std: {np.std(diff_ma_II):.4f}")
print(f"Lead II - Difference BP Filter - Mean: {np.mean(diff_bp_II):.4f}, Std: {np.std(diff_bp_II):.4f}")
print(f"Lead V5 - Difference MA Filter - Mean: {np.mean(diff_ma_V5):.4f}, Std: {np.std(diff_ma_V5):.4f}")
print(f"Lead V5 - Difference BP Filter - Mean: {np.mean(diff_bp_V5):.4f}, Std: {np.std(diff_bp_V5):.4f}")


fig, axes = plt.subplots(3, 2, figsize=(16, 12), dpi=300)

# Column 1: Lead II
# Plot 1: Original Signal - Lead II
axes[0, 0].plot(scaled_II_signal, label='Original Signal', color='blue', alpha=0.7)
axes[0, 0].set_title(f'Original Signal (Lead II) - Subject {subject_id}')
axes[0, 0].set_xlabel('Sample Index')
axes[0, 0].set_ylabel('Amplitude')
axes[0, 0].grid(True, alpha=0.3)

# Plot 2: Moving Average Filtered - Lead II
axes[1, 0].plot(mavg_filtered_II, label='Moving Average Filtered', color='red', alpha=0.7)
axes[1, 0].set_title(f'Moving Average Filtered (Lead II) - Subject {subject_id}')
axes[1, 0].set_xlabel('Sample Index')
axes[1, 0].set_ylabel('Amplitude')
axes[1, 0].grid(True, alpha=0.3)

# Plot 3: Band-Pass Filtered - Lead II
axes[2, 0].plot(bp_filtered_II, label='Band-Pass Filtered', color='green', alpha=0.7)
axes[2, 0].set_title(f'Band-Pass Filtered (Lead II) - Subject {subject_id}')
axes[2, 0].set_xlabel('Sample Index')
axes[2, 0].set_ylabel('Amplitude')
axes[2, 0].grid(True, alpha=0.3)

# Column 2: Lead V5
# Plot 1: Original Signal - Lead V5
axes[0, 1].plot(scaled_V5_signal, label='Original Signal', color='blue', alpha=0.7)
axes[0, 1].set_title(f'Original Signal (Lead V5) - Subject {subject_id}')
axes[0, 1].set_xlabel('Sample Index')
axes[0, 1].set_ylabel('Amplitude')
axes[0, 1].grid(True, alpha=0.3)

# Plot 2: Moving Average Filtered - Lead V5
axes[1, 1].plot(mavg_filtered_V5, label='Moving Average Filtered', color='red', alpha=0.7)
axes[1, 1].set_title(f'Moving Average Filtered (Lead V5) - Subject {subject_id}')
axes[1, 1].set_xlabel('Sample Index')
axes[1, 1].set_ylabel('Amplitude')
axes[1, 1].grid(True, alpha=0.3)

# Plot 3: Band-Pass Filtered - Lead V5
axes[2, 1].plot(bp_filtered_V5, label='Band-Pass Filtered', color='green', alpha=0.7)
axes[2, 1].set_title(f'Band-Pass Filtered (Lead V5) - Subject {subject_id}')
axes[2, 1].set_xlabel('Sample Index')
axes[2, 1].set_ylabel('Amplitude')
axes[2, 1].grid(True, alpha=0.3)

plt.tight_layout()
handles = [axes[0, 0].get_lines()[0], axes[1, 0].get_lines()[0], axes[2, 0].get_lines()[0]]
labels = ['Original Signal', 'Moving Average Filtered', 'Band-Pass Filtered']
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=3, frameon=True)
plt.show()

In [ ]:
# For all the subjects, use the band-pass filter and save the filtered signals
for subject_id, data in subj_dict.items():
    fs = data['fs']
    lead_II = data['II']
    lead_V5 = data['V5']
    
    # Scale the signals
    scaled_II_signal = scaler.fit_transform(lead_II.reshape(-1, 1)).flatten()
    scaled_V5_signal = scaler.fit_transform(lead_V5.reshape(-1, 1)).flatten()
    
    # Apply band-pass filter
    bp_filtered_II = band_pass_filter(scaled_II_signal, fs, lowcut=0.5, highcut=30, order=5)
    bp_filtered_V5 = band_pass_filter(scaled_V5_signal, fs, lowcut=0.5, highcut=30, order=5)
    
    # Update the dictionary with filtered signals
    subj_dict[subject_id]['bp_filtered_II'] = bp_filtered_II
    subj_dict[subject_id]['bp_filtered_V5'] = bp_filtered_V5

In [ ]:
from scipy.signal import savgol_filter

def smooth_cardiac_signal(signal, window_size=15, poly_order=3):
    """
    Smoothes the signal while preserving R-peak height.
    window_size: Must be odd. At 500Hz, 15-31 is usually good.
    poly_order: 2 or 3 is typical for cardiac data.
    """
    return savgol_filter(signal, window_size, poly_order)

# Apply for the first subject and plot one under the other
smoothed_II = smooth_cardiac_signal(bp_filtered_II, window_size=15, poly_order=3)
smoothed_V5 = smooth_cardiac_signal(bp_filtered_V5, window_size=15, poly_order=3)

fig, axes = plt.subplots(2, 2, figsize=(16, 10), dpi=300)

# Row 1: Lead II
# Band-Pass Filtered Lead II
axes[0, 0].plot(bp_filtered_II, label='Band-Pass Filtered', color='blue', alpha=0.7)
axes[0, 0].set_title(f'Band-Pass Filtered (Lead II) - Subject {subject_id}')
axes[0, 0].set_xlabel('Sample Index')
axes[0, 0].set_ylabel('Amplitude')
axes[0, 0].grid(True, alpha=0.3)

# Smoothed Lead II
axes[0, 1].plot(smoothed_II, label='Savitzky-Golay Smoothed', color='red', alpha=0.7)
axes[0, 1].set_title(f'Savitzky-Golay Smoothed (Lead II) - Subject {subject_id}')
axes[0, 1].set_xlabel('Sample Index')
axes[0, 1].set_ylabel('Amplitude')
axes[0, 1].grid(True, alpha=0.3)

# Row 2: Lead V5
# Band-Pass Filtered Lead V5
axes[1, 0].plot(bp_filtered_V5, label='Band-Pass Filtered', color='blue', alpha=0.7)
axes[1, 0].set_title(f'Band-Pass Filtered (Lead V5) - Subject {subject_id}')
axes[1, 0].set_xlabel('Sample Index')
axes[1, 0].set_ylabel('Amplitude')
axes[1, 0].grid(True, alpha=0.3)

# Smoothed Lead V5
axes[1, 1].plot(smoothed_V5, label='Savitzky-Golay Smoothed', color='red', alpha=0.7)
axes[1, 1].set_title(f'Savitzky-Golay Smoothed (Lead V5) - Subject {subject_id}')
axes[1, 1].set_xlabel('Sample Index')
axes[1, 1].set_ylabel('Amplitude')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
handles = [axes[0, 0].get_lines()[0], axes[0, 1].get_lines()[0]]
labels = ['Band-Pass Filtered', 'Savitzky-Golay Smoothed']
fig.legend(handles, labels, loc='upper center', bbox_to_anchor=(0.5, -0.02), ncol=2, frameon=True)
plt.show()

# Print the difference between the smoothed and band-pass filtered signals
diff_II = smoothed_II - bp_filtered_II
diff_V5 = smoothed_V5 - bp_filtered_V5
print(f"Lead II - Difference (Smoothed - BP Filtered) - Mean: {np.mean(diff_II):.4f}, Std: {np.std(diff_II):.4f}")
print(f"Lead V5 - Difference (Smoothed - BP Filtered) - Mean: {np.mean(diff_V5):.4f}, Std: {np.std(diff_V5):.4f}")

In [ ]:
# For each subject, apply the savitzky-golay filter and save the smoothed signals
for subject_id, data in subj_dict.items():
    bp_filtered_II = data['bp_filtered_II']
    bp_filtered_V5 = data['bp_filtered_V5']
    
    # Apply Savitzky-Golay smoothing
    smoothed_II = smooth_cardiac_signal(bp_filtered_II, window_size=15, poly_order=3)
    smoothed_V5 = smooth_cardiac_signal(bp_filtered_V5, window_size=15, poly_order=3)
    
    # Update the dictionary with smoothed signals
    subj_dict[subject_id]['smoothed_II'] = smoothed_II
    subj_dict[subject_id]['smoothed_V5'] = smoothed_V5

#### Peak detection using Pan-Tompkins algorithm

In [ ]:
# Run the Pan-Tompkins algorithm on the band-pass filtered signals to detect R-peaks
for subject_id, data in subj_dict.items():
    fs = data['fs']
    bp_filtered_II = data['bp_filtered_II']
    bp_filtered_V5 = data['bp_filtered_V5']
    
    # Apply Pan-Tompkins algorithm
    pt_processed_II = pt_algorithm(bp_filtered_II, fs)
    pt_processed_V5 = pt_algorithm(bp_filtered_V5, fs)
    
    # Detect R-peaks
    r_peaks_II = QRSpeaks(pt_processed_II, fs)
    r_peaks_V5 = QRSpeaks(pt_processed_V5, fs)
    
    # Save R-peaks indices to the dictionary
    subj_dict[subject_id]['r_peaks_II'] = r_peaks_II
    subj_dict[subject_id]['r_peaks_V5'] = r_peaks_V5

In [ ]:
# Get the seasonal period using the Pan-Tompkins detected R-peaks
for subject_id, data in subj_dict.items():
    fs = data['fs']
    r_peaks_II = data['r_peaks_II']
    
    # Calculate RR intervals in samples
    rr_intervals_samples = np.diff(r_peaks_II)
    
    # Convert RR intervals to seconds
    rr_intervals_seconds = rr_intervals_samples / fs
    
    # Calculate average heart rate (beats per minute)
    avg_rr_interval = np.mean(rr_intervals_seconds)
    
    # Calculate seasonal period in samples
    seasonal_period_samples = int(fs * avg_rr_interval)
    
    # Save seasonal period
    subj_dict[subject_id]['seasonal_period_samples'] = seasonal_period_samples

## Extracting features

In [ ]:
from scipy import stats

def extract_basic_features(signal, fs):
    """
    Extract basic statistical features from the ECG signal.
    
    Parameters:
    -----------
    signal : numpy.ndarray
        Input ECG signal
    fs : float
        Sampling frequency
        
    Returns:
    --------
    features : dict
        Dictionary containing extracted statistical features
    """
    features = {
        'avg': np.mean(signal),
        'std': np.std(signal),
        'var': np.var(signal),
        'med': np.median(signal),
        '10p': np.percentile(signal, 10),
        '25p': np.percentile(signal, 25),
        '50p': np.percentile(signal, 50),
        '75p': np.percentile(signal, 75),
        '90p': np.percentile(signal, 90),
        'iqr': np.percentile(signal, 75) - np.percentile(signal, 25),
        'cov': 1.0 * np.mean(signal) / np.std(signal) if np.std(signal) != 0 else 0,
        'skw': stats.skew(signal),
        'kur': stats.kurtosis(signal)
    }

    return features

In [ ]:
def extract_time_domain_hrv(r_peaks, fs):
    # Convert peak indices to time intervals in milliseconds
    rr_intervals = np.diff(r_peaks) / fs * 1000 
    
    # SDNN: Standard Deviation of NN intervals
    sdnn = np.std(rr_intervals)
    
    # RMSSD: Root Mean Square of Successive Differences
    diff_rr = np.diff(rr_intervals)
    rmssd = np.sqrt(np.mean(np.square(diff_rr)))
    
    # pNN50: Percentage of successive intervals differing by > 50ms
    nn50 = np.sum(np.abs(diff_rr) > 50)
    pnn50 = (nn50 / len(diff_rr)) * 100
    return {
        'SDNN': sdnn,
        'RMSSD': rmssd,
        'pNN50': pnn50
    }

In [ ]:
def extract_detailed_morphology(signal, r_peaks, fs, window_ms=600):
    """
    Extracts the average heartbeat template and key morphological metrics.
    """
    # Convert window from milliseconds to samples
    window_samples = int((window_ms / 1000) * fs)
    half_win = window_samples // 2
    
    heartbeats = []
    for r in r_peaks:
        # Ensure we don't go out of signal bounds
        if r > half_win and r < len(signal) - half_win:
            segment = signal[r - half_win : r + half_win]
            # Baseline correction: subtract the mean of the segment
            segment = segment - np.mean(segment)
            heartbeats.append(segment)
            
    if not heartbeats:
        return None

    # Calculate the Template (Ensemble Average)
    template = np.mean(heartbeats, axis=0)
    
    # Feature Extraction from Template
    # 1. R-Peak Amplitude (max value in the center)
    r_amplitude = np.max(template)
    
    # 2. QRS Duration (rough estimate by width at half-max of the R-peak)
    # Finding points where signal is > 50% of peak height
    qrs_indices = np.where(template > (r_amplitude * 0.5))[0]
    qrs_duration_ms = (len(qrs_indices) / fs) * 1000
    
    # 3. Kurtosis of the Template (measures "peakiness")
    from scipy.stats import kurtosis
    template_kurtosis = kurtosis(template)

    return {
        "template": template,
        "r_amplitude": r_amplitude,
        "qrs_duration_ms": qrs_duration_ms,
        "template_kurtosis": template_kurtosis
    }

In [ ]:
def get_amplitude(signal, indices):
    indices = np.array(indices)
    # Create an array of NaNs with the same shape
    values = np.full(indices.shape, np.nan)
    # Only look up valid indices (not NaN and within signal bounds)
    valid = ~np.isnan(indices) & (indices < len(signal)) & (indices >= 0)
    values[valid] = signal[indices[valid].astype(int)]
    return values

In [ ]:
import neurokit2 as nk
import numpy as np
import pandas as pd
from scipy import stats

def extract_ecg_features_neurokit(signal, fs, window_ms=600):
    """
    Unified pipeline to extract HRV, Heart Rate, Morphology, AND Ischemia features.
    
    Returns:
    --------
    dict
        Combined dictionary of HRV, Template Morphology, and Ischemia markers.
    """
    
    # --- STEP 1: PROCESSING ---
    # Clean the signal
    signal_clean = nk.ecg_clean(signal, sampling_rate=fs, method="neurokit")
    
    # Find R-peaks
    _, info = nk.ecg_peaks(signal_clean, sampling_rate=fs)
    r_peaks = info["ECG_R_Peaks"]
    
    if len(r_peaks) < 2:
        return None

    # --- STEP 2: HRV & HEART RATE ---
    hrv_df = nk.hrv_time(info, sampling_rate=fs, show=False)
    
    mean_nn = hrv_df['HRV_MeanNN'].values[0]
    heart_rate = 60000 / mean_nn if mean_nn != 0 else 0
    
    hrv_features = {
        'Heart_Rate': heart_rate,
        'SDNN': hrv_df['HRV_SDNN'].values[0],
        'RMSSD': hrv_df['HRV_RMSSD'].values[0],
        'pNN50': hrv_df['HRV_pNN50'].values[0]
    }

    # --- STEP 3: TEMPLATE MORPHOLOGY (Global Shape) ---
    window_sec = window_ms / 1000
    epochs = nk.ecg_segment(signal_clean, rpeaks=r_peaks, sampling_rate=fs, 
                            show=False, tmin=-window_sec/2, tmax=window_sec/2)
    
    # Create Template (Ensemble Average)
    df_epochs = pd.concat([epochs[i]['Signal'] for i in epochs], axis=1)
    template = df_epochs.mean(axis=1).values
    
    # Standard Template Features
    r_amplitude = np.max(template)
    template_kurtosis = stats.kurtosis(template)
    
    # QRS Duration (FWHM approach on template)
    threshold = r_amplitude * 0.5
    above_threshold_indices = np.where(template > threshold)[0]
    qrs_duration_ms = ((above_threshold_indices[-1] - above_threshold_indices[0]) / fs * 1000) if len(above_threshold_indices) > 0 else np.nan

    morphology_features = {
        "template": template,
        "r_amplitude": r_amplitude,
        "qrs_duration_ms": qrs_duration_ms,
        "template_kurtosis": template_kurtosis
    }

    # --- STEP 4: ISCHEMIA FEATURES (Wave-Specific) ---
    try:
        # Use available keys, fallback to NaN if missing
        _, waves = nk.ecg_delineate(signal_clean, r_peaks, sampling_rate=fs, method="dwt")

        # Helper to safely get wave indices or NaN array
        def get_wave(waves, key, n):
            return np.array(waves[key]) if key in waves else np.full(n, np.nan)

        n_beats = len(r_peaks)
        print(f"Number of beats for ischemia feature extraction: {n_beats}")
        idx_q_onset = get_wave(waves, "ECG_Q_Peaks", n_beats)
        idx_t_peak = get_wave(waves, "ECG_T_Peaks", n_beats)
        idx_t_offset = get_wave(waves, "ECG_T_Offsets", n_beats)
        idx_s_offset = get_wave(waves, "ECG_S_Offsets", n_beats)

        # Amplitude calculations (use helper from notebook if present)
        def get_amplitude(signal, indices):
            indices = np.array(indices)
            values = np.full(indices.shape, np.nan)
            valid = ~np.isnan(indices) & (indices < len(signal)) & (indices >= 0)
            values[valid] = signal[indices[valid].astype(int)]
            return values

        baseline_amp = get_amplitude(signal_clean, idx_q_onset)
        #print(f"baseline_amp calculated: {baseline_amp}")
        t_peak_amp = get_amplitude(signal_clean, idx_t_peak)
        #print(f"t_peak_amp calculated: {t_peak_amp}")
        j_point_amp = get_amplitude(signal_clean, idx_s_offset)
        #print(f"j_point_amp calculated: {j_point_amp}")

        t_wave_amplitude = t_peak_amp - baseline_amp
        #print(f"t_wave_amplitude calculated: {t_wave_amplitude}")
        st_deviation = j_point_amp - baseline_amp
        #print(f"st_deviation calculated: {st_deviation}")

        qt_interval_sec = (idx_t_offset - idx_q_onset) / fs
        #print(f"qt_interval_sec calculated: {qt_interval_sec}")

        rr_intervals_samples = np.diff(r_peaks, prepend=np.nan)
        #print(f"rr_intervals_samples calculated: {rr_intervals_samples}")   
        rr_intervals_sec = rr_intervals_samples / fs
        #print(f"rr_intervals_sec calculated: {rr_intervals_sec}")
        qtc_bazett = qt_interval_sec / np.sqrt(rr_intervals_sec)
        #print(f"qtc_bazett calculated: {qtc_bazett}")

        results = pd.DataFrame({
            "R_Peak_Index": r_peaks,
            "T_Wave_Amplitude": t_wave_amplitude,
            "ST_Deviation": st_deviation,
            "QT_Interval_Sec": qt_interval_sec,
            "RR_Interval_Sec": rr_intervals_sec,
            "QTc_Bazett": qtc_bazett
        })

        def safe_nanmean(arr):
            arr = np.asarray(arr)
            if arr.size == 0 or np.all(np.isnan(arr)):
                return np.nan
            return np.nanmean(arr)

        ischemia_features = {
            "mean_t_wave_amplitude": safe_nanmean(results["T_Wave_Amplitude"].values),
            "mean_st_deviation": safe_nanmean(results["ST_Deviation"].values),
            "mean_qt_interval_sec": safe_nanmean(results["QT_Interval_Sec"].values),
            "mean_qtc_bazett": safe_nanmean(results["QTc_Bazett"].values)
        }
    except Exception as e:
        print(f"Ischemia feature extraction failed: {e}")
        ischemia_features = {
            "mean_t_wave_amplitude": np.nan,
            "mean_st_deviation": np.nan,
            "mean_qt_interval_sec": np.nan,
            "mean_qtc_bazett": np.nan
        }

    # Combine all features
    all_features = {}
    all_features.update(hrv_features)
    all_features.update(morphology_features)
    all_features.update(ischemia_features)
    return all_features

In [ ]:
import pandas as pd

# Assuming you have a dictionary or list of your data
# data_structure = {'Subj_01': signal_array_1, 'Subj_02': signal_array_2, ...}

scalar_results = []   # For SDNN, RMSSD, Kurtosis, etc.
templates_dict = {}   # Store templates separately (as they are arrays)

# 1. Loop through subjects
for subj_id, data in subj_dict.items():
    signal = data['smoothed_II']
    fs = data['fs']
    
    # 2. Call the function
    # Note: Ensure 'fs' matches your specific device (e.g., 250, 500, 1000)
    features = extract_ecg_features_neurokit(signal, fs=fs, window_ms=600)
    
    # 3. Check for Valid Output (Handle None)
    if features is not None:
        # Separate the Template (Array) from the Scalars (Numbers)
        # We pop it out so it doesn't mess up the pandas DataFrame later
        template_array = features.pop('template')
        
        # Store the template separately
        templates_dict[subj_id] = template_array
        
        # Add the ID to the scalar features for tracking
        features['subject_id'] = subj_id
        features['smoothed_II'] = signal 
        scalar_results.append(features)
        
    else:
        print(f"Skipping {subj_id}: Signal too short or noisy.")

# 4. Create your Final DataFrame for Classification
features_df = pd.DataFrame(scalar_results)

#Make sure Subject_ID is the first column
cols = features_df.columns.tolist()
cols.insert(0, cols.pop(cols.index('subject_id')))
features_df = features_df[cols]

features_df

In [ ]:
hd_df = pd.read_csv('../../data/processed/hd.csv')

# Drop st_deviation column due to high missingness
features_df = features_df.drop(columns=['mean_st_deviation'])

# Drop the rows with mean_qt_interval_sec missing values and mean_qtc_bazett missing values
features_df = features_df.dropna(subset=['mean_qt_interval_sec', 'mean_qtc_bazett'])

features_df['subject_id'] = features_df['subject_id'].astype(int)
full_df = pd.merge(features_df, hd_df, on='subject_id', how='inner')

# Save as pickle to preserve numpy arrays
full_df.to_pickle('../../data/processed/time_series_features.pkl')

full_df